In [1]:
import json, os
import numpy as np
import pandas as pd

pre = pd.read_csv("../data/processed/preprocessed_data.csv")  # has type column ideally
eng = pd.read_csv("../data/processed/feature_engineered_data.csv")  # final training style

print("pre:", pre.shape)
print("eng:", eng.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/preprocessed_data.csv'

In [2]:
import os, glob

print("CWD:", os.getcwd())
print("data folder exists?", os.path.exists("../data"))
print("processed exists?", os.path.exists("../data/processed"))

print("\nFiles in ../data (first 30):")
print(glob.glob("../data/**/*", recursive=True)[:30])

CWD: E:\transaction-fraud-intelligence\notebooks
data folder exists? True
processed exists? True

Files in ../data (first 30):
['../data\\processed', '../data\\processed\\best_model_probs.npy', '../data\\processed\\X_test.csv', '../data\\processed\\y_test.csv']


In [3]:
import glob

raw_files = glob.glob("../data/raw/*.csv")
raw_files

[]

In [ ]:
config = {}

config["large_threshold_p95"] = float(pre["amount"].quantile(0.95))
config["median_oldbalanceDest"] = float(pre["oldbalanceDest"].median())

max_step = int(pre["step"].max())
config["step_bins"] = np.linspace(0, max_step, 7).tolist()

step_counts = pre.groupby("step").size().to_dict()
config["step_counts"] = {str(int(k)): int(v) for k, v in step_counts.items()}
config["median_transactions_in_step"] = float(np.median(list(step_counts.values())))

config


os.makedirs("../models", exist_ok=True)

with open("../models/feature_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved: ../models/feature_config.json")

In [1]:
import json, os
import numpy as np
import pandas as pd

X_test = pd.read_csv("../data/processed/X_test.csv")

config = {}
config["large_threshold_p95"] = float(X_test["amount"].quantile(0.95))
config["median_oldbalanceDest"] = float(X_test["oldbalanceDest"].median())

max_step = int(X_test["step"].max())
config["step_bins"] = np.linspace(0, max_step, 7).tolist()

step_counts = X_test.groupby("step").size().to_dict()
config["step_counts"] = {str(int(k)): int(v) for k, v in step_counts.items()}
config["median_transactions_in_step"] = float(np.median(list(step_counts.values())))

os.makedirs("../models", exist_ok=True)

with open("../models/feature_config.json", "w") as f:
    json.dump(config, f, indent=2)

with open("../models/feature_columns.json", "w") as f:
    json.dump(X_test.columns.tolist(), f, indent=2)

print("DONE ✅ feature_config.json + feature_columns.json created")

DONE ✅ feature_config.json + feature_columns.json created


In [2]:
import os
print(os.path.exists("../models/feature_config.json"))
print(os.path.exists("../models/feature_columns.json"))

True
True


In [3]:
import glob, os

raw_files = glob.glob("../data/raw/**/*.csv", recursive=True)
print("Found raw CSVs:", len(raw_files))
raw_files[:5]

Found raw CSVs: 1


['../data/raw\\transaction_data.csv']

In [4]:
import pandas as pd
import numpy as np

RAW_PATH = raw_files[0]   # change if needed
print("Using:", RAW_PATH)

usecols = ["step", "amount", "oldbalanceDest"]
dtypes = {"step":"int32", "amount":"float32", "oldbalanceDest":"float32"}

raw = pd.read_csv(RAW_PATH, usecols=usecols, dtype=dtypes)
raw.shape

Using: ../data/raw\transaction_data.csv


(6362620, 3)

In [5]:
import json, os

config = {}
config["large_threshold_p95"] = float(raw["amount"].quantile(0.95))
config["median_oldbalanceDest"] = float(raw["oldbalanceDest"].median())

max_step = int(raw["step"].max())
config["step_bins"] = np.linspace(0, max_step, 7).tolist()

step_counts = raw.groupby("step").size().to_dict()
config["step_counts"] = {str(int(k)): int(v) for k, v in step_counts.items()}
config["median_transactions_in_step"] = float(np.median(list(step_counts.values())))

os.makedirs("../models", exist_ok=True)
with open("../models/feature_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved: ../models/feature_config.json")

Saved: ../models/feature_config.json
